# Hybrid Search RAG — Keyword + Semantic + Reranking (2 of 6)

## RAG Workshop Series

This notebook is part of a 6-notebook series (split from the original `RAG.ipynb`), each one runnable on its own in Google Colab:

1. **`basic_RAG.ipynb`** — naive vector RAG: chunk → embed → cosine similarity → prompt → LLM
2. **`hybrid_search_RAG.ipynb`** — BM25 keyword search + semantic search fusion + cross-encoder reranking
3. **`query_and_chunking_RAG.ipynb`** — query rewriting, advanced chunking strategies, metadata filtering
4. **`agentic_RAG.ipynb`** — Corrective RAG (CRAG), Adaptive RAG (routing), Agentic RAG (ReAct loop)
5. **`pdf_chroma_RAG.ipynb`** — build RAG over a real PDF, store vectors persistently in Chroma
6. **`rag_when_to_use.ipynb`** — reference: when RAG is (and isn't) the right tool

Each notebook installs its own dependencies and rebuilds whatever context it needs, so you can open any one directly without running the others first.

## Recap

In `basic_RAG.ipynb` we built naive vector (semantic) search: embed chunks, embed the question, rank by cosine similarity.

That approach struggles with **exact terms** — product codes, error codes, acronyms — where a keyword match is more reliable than a semantic one. This notebook adds:

- **BM25** — classic keyword ranking
- **Hybrid search** — combine semantic + BM25 scores
- **Reranking** — a cross-encoder that re-scores the top candidates for precision

## Setup

In [ ]:
!pip install -q sentence-transformers rank_bm25

### Knowledge base (same AWS services corpus as `basic_RAG.ipynb`)

In [ ]:
documents = [
    """
    Amazon S3 is an object storage service designed for storing
    and retrieving files. It provides high durability and is
    commonly used for backups, static websites, data lakes,
    and application assets.
    """,

    """
    Amazon SQS is a managed message queue service.

    It allows applications to communicate asynchronously.

    Producers send messages to a queue and consumers process
    those messages independently.

    SQS is commonly used to decouple distributed applications.
    """,

    """
    AWS Lambda is a serverless compute service.

    Developers upload code and AWS executes the code in response
    to events.

    Lambda automatically manages servers and scales applications
    based on incoming requests.
    """,

    """
    Amazon DynamoDB is a managed NoSQL database.

    It provides low-latency access to data and automatically
    scales to handle large workloads.
    """
]

### Chunking

In [ ]:
def chunk_text(text, chunk_size=250):
    words = text.split()

    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks


chunks = []

for document in documents:
    chunks.extend(chunk_text(document))


print("Number of chunks:", len(chunks))

for index, chunk in enumerate(chunks):
    print(f"\nCHUNK {index}")
    print(chunk)

### Embeddings

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
%%capture
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
chunk_embeddings = embedding_model.encode(chunks)

print(chunk_embeddings.shape)

# Step 7: Keyword Search BM25


## What is BM25? It's a keyword based ranking algorithm (Best Matching 25 Algorithm)

## Core Components of BM25

- **Term Frequency (TF):** Measures how often a query word appears in a document. Repeated occurrences eventually have diminishing impact.
- **Inverse Document Frequency (IDF):** Gives higher scores to rare words and lower scores to common words.
- **Document Length Normalization:** Prevents longer documents from automatically scoring higher simply because they contain more words.


In [ ]:
from rank_bm25 import BM25Okapi

# Tokenize each chunk
tokenized_chunks = [
    chunk.lower().split()
    for chunk in chunks
]

# Create the BM25 index
bm25 = BM25Okapi(tokenized_chunks)

In [ ]:
def keyword_search(question, top_k=2):

    # Tokenize the question
    query_tokens = question.lower().split()

    # Calculate BM25 scores
    scores = bm25.get_scores(query_tokens)

    # Get the top-k chunks
    top_indices = scores.argsort()[::-1][:top_k]

    results = []

    for index in top_indices:
        results.append({
            "text": chunks[index],
            "bm25_score": float(scores[index])
        })

    return results

In [ ]:
question = "Which AWS service can help decouple applications?"

results = keyword_search(question, top_k=4)

for result in results:
    print(
        round(result["bm25_score"], 3),
        result["text"][:150]
    )

#STEP 8: Hybrid Search

                    Query
                      │
            ┌─────────┴─────────┐
            ↓                   ↓
     Semantic Search       Keyword Search
            │                   │
     Cosine Similarity        BM25
            │                   │
    Semantic Score          BM25 Score
            │                   │
            └─────────┬─────────┘
                      ↓
               Hybrid Search

In [ ]:
def hybrid_search(question, top_k=2, semantic_weight=0.5, keyword_weight=0.5):

    # -------------------------
    # 1. Semantic Search
    # -------------------------

    question_embedding = embedding_model.encode([question])

    semantic_scores = cosine_similarity(
        question_embedding,
        chunk_embeddings
    )[0]

    # -------------------------
    # 2. Keyword Search (BM25)
    # -------------------------

    query_tokens = question.lower().split()

    bm25_scores = bm25.get_scores(query_tokens)

    # -------------------------
    # 3. Normalize Scores
    # -------------------------

    semantic_normalized = (
        semantic_scores - semantic_scores.min()
    ) / (
        semantic_scores.max() - semantic_scores.min()
    )

    bm25_normalized = (
        bm25_scores - bm25_scores.min()
    ) / (
        bm25_scores.max() - bm25_scores.min()
    )

    # -------------------------
    # 4. Combine Scores
    # -------------------------

    hybrid_scores = (
        semantic_weight * semantic_normalized
        + keyword_weight * bm25_normalized
    )

    # -------------------------
    # 5. Get Top Results
    # -------------------------

    top_indices = hybrid_scores.argsort()[::-1][:top_k]

    results = []

    for index in top_indices:
        results.append({
            "text": chunks[index],
            "semantic_score": float(semantic_normalized[index]),
            "bm25_score": float(bm25_normalized[index]),
            "hybrid_score": float(hybrid_scores[index])
        })

    return results

In [ ]:
question = "Which AWS service can help decouple applications?"

results = hybrid_search(question, top_k=4)

for result in results:
    print(
        f"Hybrid: {result['hybrid_score']:.3f} | "
        f"Semantic: {result['semantic_score']:.3f} | "
        f"BM25: {result['bm25_score']:.3f}"
    )
    print(result["text"][:150])
    print()

let's change the dataset and see if hybrid works?

In [ ]:
documents = [
    """
    The application becomes unresponsive when users upload large files.
    The server runs out of memory during the upload process.
    """,

    """
    Users are unable to authenticate after enabling two-factor authentication.
    The login process fails with an invalid verification code error.
    """,

    """
    The API returns error code E403 when clients attempt to access protected
    resources. Check the API credentials and authorization permissions.
    """,

    """
    Database queries are taking several seconds to complete.
    Adding indexes to frequently queried columns can improve database performance.
    """,

    """
    The background worker stops processing jobs when the message queue
    contains a large number of pending tasks.
    """
]

In [ ]:
from rank_bm25 import BM25Okapi

def process_documents(documents, embedding_model, chunk_size=250):
    # Chunk the documents
    all_chunks = []
    for document in documents:
        all_chunks.extend(chunk_text(document, chunk_size))

    # Generate chunk embeddings
    chunk_embeddings = embedding_model.encode(all_chunks)

    # Tokenize chunks for BM25
    tokenized_chunks = [
        chunk.lower().split()
        for chunk in all_chunks
    ]
    bm25_index = BM25Okapi(tokenized_chunks)

    return all_chunks, chunk_embeddings, bm25_index

# Now, call the new function to process the current documents
chunks, chunk_embeddings, bm25 = process_documents(documents, embedding_model)

In [ ]:
question = "Why is the application running out of memory?"
results = hybrid_search(question, top_k=4)

for result in results:
    print(
        f"Hybrid: {result['hybrid_score']:.3f} | "
        f"Semantic: {result['semantic_score']:.3f} | "
        f"BM25: {result['bm25_score']:.3f}"
    )
    print(result["text"][:150])
    print()

## Step 9: Reranking

Hybrid search gives us candidate documents.

But retrieval doesn't necessarily give us the best ordering since they are bi-encoders.

A reranker takes the query and retrieved candidates and
scores how relevant each candidate is to that specific query.

Query + Document → Cross-Encoder → Relevance Score


Because the model sees the query and document together, it can make a much more detailed relevance judgment.

| Reranker          | Type                 | Typical use                                            |
| ----------------- | -------------------- | ------------------------------------------------------ |
| **Cohere Rerank** | Hosted API           | Enterprise teams wanting managed infrastructure        |
| **BGE Reranker**  | Open-weight          | Self-hosted RAG, very popular open-source choice       |
| **Jina Reranker** | Hosted / open models | Enterprise RAG, multilingual/document-heavy workloads  |
| **Voyage Rerank** | Hosted API           | High-quality managed retrieval                         |
| **Qwen Reranker** | Open-weight          | Newer, larger-model option for quality-focused systems |


LLMs can also used are rerankers, especially small LLMs if cost matters less than accuracy  

Why not use cross-encoders instead of vector search?

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L6-v2"
)

In [ ]:
def rerank(question, results, top_k=2):

    # Create query-document pairs
    pairs = [
        (question, result["text"])
        for result in results
    ]

    # Calculate relevance scores
    scores = reranker.predict(pairs)

    # Add scores to results
    for result, score in zip(results, scores):
        result["reranker_score"] = float(score)

    # Sort by reranker score
    results = sorted(
        results,
        key=lambda x: x["reranker_score"],
        reverse=True
    )

    return results[:top_k]

In [ ]:
documents = [
    """
    Amazon S3 is an object storage service designed for storing
    and retrieving files. It provides high durability and is
    commonly used for backups, static websites, data lakes,
    and application assets.
    """,

    """
    Amazon SQS is a managed message queue service.

    It allows applications to communicate asynchronously.

    Producers send messages to a queue and consumers process
    those messages independently.

    SQS is commonly used to decouple distributed applications.
    """,

    """
    AWS Lambda is a serverless compute service.

    Developers upload code and AWS executes the code in response
    to events.

    Lambda automatically manages servers and scales applications
    based on incoming requests.
    """,

    """
    Amazon DynamoDB is a managed NoSQL database.

    It provides low-latency access to data and automatically
    scales to handle large workloads.
    """
]
# FIXED: Pass 'embedding_model' instead of the conflicting 'model' variable
chunks, chunk_embeddings, bm25 = process_documents(documents, embedding_model)

In [ ]:
question = "Which AWS service can help decouple applications?"

candidates = hybrid_search(
    question,
    top_k=4
)

results = rerank(
    question,
    candidates,
    top_k=2
)

for result in results:
    print(
        f"Reranker: {result['reranker_score']:.3f}"
    )
    print(result["text"][:150])
    print()

---

➡️ **Next:** [`query_and_chunking_RAG.ipynb`](./query_and_chunking_RAG.ipynb) — rewrite queries before retrieval, try smarter chunking strategies, and filter by metadata.